# Finetuning Qwen3-ASR (Supervised SFT)

In [ ]:
!pip install -U "transformers>=5.13.0" accelerate torch torchaudio torchvision jiwer pandas tqdm torchcodec bitsandbytes

## Setup

In [ ]:
import torch
from pathlib import Path
from transformers import AutoModelForMultimodalLM, AutoProcessor

model_id = "Qwen/Qwen3-ASR-1.7B-hf"
MANIFEST_DIR = Path("/training-1/asr_finetune")   # <-- adapt
OUTPUT_DIR = MANIFEST_DIR / "checkpoint"

processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(model_id, device_map="auto", dtype=torch.bfloat16)
print(f"Model loaded on {model.device} with dtype {model.dtype}")

# ~1.7B params in bf16 (~4 GB) + gradients + 8-bit optimizer states fit in 10 GB VRAM
model.config.use_cache = False
model.gradient_checkpointing_enable()

## Load the Training and Dev Manifests

In [ ]:
import json

MANIFEST_TRAIN = MANIFEST_DIR / "manifest_train.jsonl"
MANIFEST_DEV = MANIFEST_DIR / "manifest_dev.jsonl"

def read_manifest(path):
    """Read a JSONL manifest into a list of dicts."""
    with open(path, encoding="UTF-8") as rf:
        return [json.loads(line) for line in rf]

train_data = read_manifest(MANIFEST_TRAIN)
dev_data = read_manifest(MANIFEST_DEV)

print(len(train_data), "train samples,", len(dev_data), "dev samples")
print(train_data[0])

## Collate Function (Audio + Prompt + Target Text)

In [ ]:
def collate_fn(batch):
    """Build model inputs from manifest entries via apply_transcription_request."""
    input_ids_list, labels_list = [], []
    for sample in batch:
        kwargs = {"audio": sample["audio"]}
        if sample.get("prompt"):
            kwargs["prompt"] = sample["prompt"]
        inputs = processor.apply_transcription_request(**kwargs).to(model.device, model.dtype)

        # Append the target transcript as supervision and mask the prompt part
        target_ids = processor.tokenizer(sample["text"], add_special_tokens=False,
                                         return_tensors="pt").input_ids[0].to(model.device)
        full_ids = torch.cat([inputs["input_ids"][0], target_ids])
        labels = full_ids.clone()
        labels[: inputs["input_ids"].shape[1]] = -100   # ignore prompt/audio tokens

        input_ids_list.append(full_ids)
        labels_list.append(labels)

    # NOTE: adapting audio features + batching to the Qwen3-ASR processor API may require
    # adjustments — check the model card / transformers docs for the training recipe.
    return {
        "input_ids": torch.nn.utils.rnn.pad_sequence(input_ids_list, batch_first=True),
        "labels": torch.nn.utils.rnn.pad_sequence(labels_list, batch_first=True,
                                                  padding_value=-100),
        **{k: v for k, v in inputs.items() if k not in ("input_ids",)},
    }

## Training Configuration

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR.as_posix(),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=1e-5,
    num_train_epochs=3,
    eval_strategy="epoch",
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    gradient_checkpointing=True,
    optim="paged_adamw_8bit",   # keeps optimizer states small enough for 10 GB VRAM
    report_to=[],
)

## Evaluation (Dev WER)

In [ ]:
import jiwer

def compute_dev_wer():
    """Greedy-transcribe the dev set with the current model and compute WER."""
    model.eval()
    wers = []
    for sample in dev_data:
        kwargs = {"audio": sample["audio"]}
        if sample.get("prompt"):
            kwargs["prompt"] = sample["prompt"]
        inputs = processor.apply_transcription_request(**kwargs).to(model.device, model.dtype)
        with torch.inference_mode():
            output_ids = model.generate(**inputs, max_new_tokens=256)
            generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
            hyp = processor.decode(generated_ids, return_format="transcription_only")[0].strip()
        wers.append(jiwer.wer(sample["text"], hyp))
    model.train()
    return sum(wers) / len(wers)

print("Dev WER before training:", compute_dev_wer())

## Train

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=dev_data,
    data_collator=collate_fn,
)

trainer.train()

## Write Out the Checkpoint

In [ ]:
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

print(f"Checkpoint written to {OUTPUT_DIR}")